In [1]:
!pip install pyspark datasets huggingface_hub -q

In [2]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)

In [3]:
!wget -O data.gz "https://go.criteo.net/criteo-research-kaggle-display-advertising-challenge-dataset.tar.gz"

--2026-04-04 01:59:23--  https://go.criteo.net/criteo-research-kaggle-display-advertising-challenge-dataset.tar.gz
Resolving go.criteo.net (go.criteo.net)... 74.119.118.85, 2620:100:a005::25
Connecting to go.criteo.net (go.criteo.net)|74.119.118.85|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://criteostorage.blob.core.windows.net/criteo-research-datasets/kaggle-display-advertising-challenge-dataset.tar.gz [following]
--2026-04-04 01:59:24--  https://criteostorage.blob.core.windows.net/criteo-research-datasets/kaggle-display-advertising-challenge-dataset.tar.gz
Resolving criteostorage.blob.core.windows.net (criteostorage.blob.core.windows.net)... 20.209.1.1
Connecting to criteostorage.blob.core.windows.net (criteostorage.blob.core.windows.net)|20.209.1.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4576820670 (4.3G) [application/x-gzip]
Saving to: ‘data.gz’

data.gz             100%[===================>]   4.26G  12.2M

In [4]:
!tar -zxvf /content/data.gz

tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
readme.txt
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.creationtime'
tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
test.txt
tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
train.txt


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Criteo_MMDS_Benchmarking") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


In [6]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

fields = [StructField("label", IntegerType(), True)]

for i in range(1, 14):
    fields.append(StructField(f"I{i}", IntegerType(), True))

for i in range(1, 27):
    fields.append(StructField(f"C{i}", StringType(), True))

criteo_schema = StructType(fields)

print(len(criteo_schema.fields))

40


In [7]:
# df_raw = spark.read.csv(f"{raw_data_dir}/*.gz", sep="\t", schema=criteo_schema)
df_raw = spark.read.csv("train.txt", sep="\t", schema=criteo_schema)
print(f"Num records: {df_raw.count()}")

Num records: 45840617


In [8]:
from pyspark.ml.feature import Imputer

num_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

df_clean_cat = df_raw.fillna("unknown", subset=cat_cols)

imputer = Imputer(
    inputCols=num_cols,
    outputCols=num_cols,
    strategy="median"
)

imputer_model = imputer.fit(df_clean_cat)
df_clean = imputer_model.transform(df_clean_cat)

df_clean.show(10)

+-----+---+---+---+---+-----+---+---+---+---+---+---+---+---+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+
|label| I1| I2| I3| I4|   I5| I6| I7| I8| I9|I10|I11|I12|I13|      C1|      C2|      C3|      C4|      C5|      C6|      C7|      C8|      C9|     C10|     C11|     C12|     C13|     C14|     C15|     C16|     C17|     C18|     C19|     C20|     C21|     C22|     C23|     C24|     C25|     C26|
+-----+---+---+---+---+-----+---+---+---+---+---+---+---+---+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+
|    0|  1|  1|  5|  0| 1382|  4| 15|  2|181|  1|  2|  0|  2|68fd1e64|80e26c9b|fb936136|7b4723c4|25c83c98|7e0ccc

In [2]:
output_parquet_dir = "./criteo_cleaned_parquet"

# Dùng coalesce(50) để chia thành 50 file lớn thay vì hàng nghìn file nhỏ
df_clean.coalesce(50).write.mode("overwrite").parquet(output_parquet_dir)

print("Đã hoàn tất chuyển đổi và lưu file Parquet!")

In [3]:
from datasets import load_dataset

# Đọc toàn bộ các file parquet trong thư mục Spark vừa tạo ra
# Các file dữ liệu của Spark thường bắt đầu bằng "part-"
hf_dataset = load_dataset(
    "parquet",
    data_files=f"{output_parquet_dir}/part-*.parquet",
    split="train",
    streaming=True
)

# Khai báo tên repo của bạn (thay 'username' bằng tài khoản của bạn)
repo_id = "nmpogg/criteo-cleaned"

# Đẩy lên Hugging Face
hf_dataset.push_to_hub(repo_id)

print(f"Đã đẩy dữ liệu thành công lên: https://huggingface.co/datasets/{repo_id}")

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/50 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/556 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/578 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   5%|5         | 3.73MB / 73.0MB            

Đã đẩy dữ liệu thành công lên: https://huggingface.co/datasets/nmpogg/criteo-cleaned
